# blunderDB — lire sa base dans un notebook

Ce carnet ne lit que les exports CSV de `blunderdb list`. Rien d'autre : pas
d'accès direct au fichier `.db`, pas de format propriétaire, pas de dépendance
à une version de schéma. Ce que vous voyez ici, vous pouvez le refaire sur
votre propre base — et ce qu'il ne fait pas, vous l'écrivez dans la cellule
suivante.

```bash
blunderdb list --db votre.db --type positions --format csv > positions.csv
blunderdb list --db votre.db --type moves     --format csv > moves.csv
blunderdb list --db votre.db --type analyses  --format csv > analyses.csv
```

In [ ]:
import pathlib
import pandas as pd

HERE = pathlib.Path.cwd()
REQUIRED = ["positions.csv", "moves.csv", "analyses.csv"]

missing = [name for name in REQUIRED if not (HERE / name).exists()]
if missing:
    raise SystemExit(
        "Fichiers manquants : " + ", ".join(missing) + "\n"
        "Exportez-les d'abord (voir la cellule précédente)."
    )

positions = pd.read_csv(HERE / "positions.csv")
moves = pd.read_csv(HERE / "moves.csv")
analyses = pd.read_csv(HERE / "analyses.csv")

print(f"{len(positions)} positions, {len(moves)} coups, {len(analyses)} analyses")

## Le PR dans le temps

Le Performance Rating est `500 x somme des erreurs (en points d'équité) /
nombre de décisions comptées`. Les erreurs sont stockées en **millipoints**,
d'où la division par 1000.

Ce calcul est volontairement le plus simple possible : il compte toutes les
décisions ayant une erreur, sans le prédicat XG des décisions « comptées »
(coups forcés exclus, videaux serrés seulement). blunderDB applique ce prédicat
dans ses propres statistiques ; ici, l'intérêt est de montrer la formule à nu.

In [ ]:
decisions = moves.merge(
    positions[["position_id", "decision_type", "played_move_error_mp", "cube_error_mp"]],
    on="position_id",
    how="left",
)
decisions["error_mp"] = decisions.apply(
    lambda r: r["cube_error_mp"] if r["decision_type"] == "cube" else r["played_move_error_mp"],
    axis=1,
).fillna(0)

by_match = decisions.groupby(["match_id", "match_date"]).agg(
    decisions=("error_mp", "size"),
    sum_error_mp=("error_mp", "sum"),
).reset_index()
by_match["pr"] = 500 * by_match["sum_error_mp"] / 1000 / by_match["decisions"]
by_match = by_match.sort_values("match_date")
by_match[["match_date", "decisions", "pr"]]

In [ ]:
ax = by_match.plot(x="match_date", y="pr", marker="o", legend=False, figsize=(8, 3))
ax.set_xlabel("date du match")
ax.set_ylabel("PR")
ax.set_title("Performance Rating par match")
ax.figure.tight_layout()

## Distribution des magnitudes d'erreur

La forme de la courbe en dit plus que la moyenne : beaucoup de petites erreurs
et peu de grosses n'est pas le même joueur que l'inverse, à PR égal.

In [ ]:
errors = decisions.loc[decisions["error_mp"] > 0, "error_mp"] / 1000
ax = errors.plot.hist(bins=30, figsize=(8, 3))
ax.set_xlabel("erreur (points d'équité)")
ax.set_ylabel("décisions")
ax.set_title("Distribution des magnitudes d'erreur")
ax.figure.tight_layout()
print(errors.describe())

## Les pires décisions

Avec leur XGID : collez-en un dans blunderDB (`import XGID=…`) ou dans eXtreme
Gammon pour retrouver la position.

In [ ]:
worst = (
    decisions.sort_values("error_mp", ascending=False)
    .head(10)
    .merge(positions[["position_id", "xgid", "phase"]], on="position_id", how="left")
)
worst["error"] = worst["error_mp"] / 1000
worst[["match_date", "player1", "player2", "decision_type", "phase", "error", "xgid"]]

## Et ensuite

Tout est dans trois DataFrames. Quelques pistes :

* le PR par phase de partie (`positions.phase`) : `decisions.groupby("phase")` ;
* la chance par match (`moves.luck_mp`), qui est mesurée par le logiciel
  d'origine et vaut `NaN` quand le format ne la transporte pas — ne la
  confondez pas avec zéro ;
* les positions revenues plusieurs fois : `moves.position_id.value_counts()`,
  puisqu'une position est identifiée par sa structure et dédupliquée entre les
  matchs.